PHẢI NHỚ: luôn apply mask cả 2 phía dấu =
THỰC TẾ:  FILLNA() THAY VÌ MASK MỦNG LINH TINH

raw = raw_1.fillna(raw_2).fillna(N lần)
------------------------------ 2 kiểu kết quả như nhau (cứ nghĩ fillna = lấy những mảng TRUE lắp vào nhau)
mask = raw.isna()
raw.loc[mask] = raw.loc[mask].function

------------------------------ 
UPDATE tư duy cleaning pipeline:
Chỉ Fillna trước Astype đối với 'int' + 'float'

In [84]:
import pandas as pd
import numpy as np
# pd.options.display.float_format = '{:,.0f}'.format

# df = pd.read_csv(r'D:\Python\LMHN_2025_DIRTY.csv')
df = pd.read_csv(r'D:\Python\James LMHN - 2026 - RAW.csv')
# Standardize cols names
df.columns = [col.strip().replace('\r', '').replace('\n', '').replace(' ', '_').lower() for col in df.columns]

# Get col name from regex
def get_col(df, rule):
  # col_mask = object index[boolean] 
    col_mask = df.columns.astype(str).str.contains(rule, case=False, regex=True)
  # count_true = Count True in col_mask
    count_true = col_mask.sum()
  # Apply mask to columns_names [object]
    col = df.columns[col_mask]
     
    if   count_true == 1:     
        return col[0]
    elif count_true == 0:
        print(f'get_col 0 match for regex: {rule}')
        return None
    else:
        print(f'get_col return first_match for regex: {rule}.Found {count_true} cols: {col.to_list()} Investigate later!')
        return col[0]

# Nhóm cột tự động hóa 
date = get_col(df, r'\bdate\b|\bngay\b|\bday\b')
time = get_col(df, r'\btime\b|\bgio\b|\bhour\b')
price = get_col(df, r'\bprice\b')
# Nhóm manual
qty = get_col(df, r'\bqty\b|quantity|sl')
ins_stt = get_col(df, r'ins_stt|tg')

special = (qty, ins_stt)
special_cols = [s for s in special if s is not None]

# Drop unnecessary columns/rows. Never sort when cleaning, don't mess with labels
drop_cols = ['ins_fee', 'disc_percent', 'disc_amount', 'vat', 'disc%', 'disc$', 'đết', 'note', 'ad']
df = df.drop(drop_cols, axis=1, errors='ignore')


#! Need to convert date/time/string/boolean cols BEFORE Fillna()
# Date processing | pd.to_datetime(df['col_name']) |
if date:
  d1 = pd.to_datetime(df[date], format='%Y-%m-%d', errors='coerce')    # 2026-01-20
  d2 = pd.to_datetime(df[date], format='%d-%m-%Y', errors='coerce')    # 20-01-2026
  df[date] = d1.fillna(d2)
# Time processing
if time:
  t1 = pd.to_datetime(df[time], format='%I:%M%p' , errors='coerce')    # 8:30am
  t2 = pd.to_datetime(df[time], format='%H:%M:%S', errors='coerce')    # 20:30:00
  t3 = pd.to_datetime(df[time], format='%H:%M'   , errors='coerce')    # 16:20
  df[time] = t1.fillna(t2).fillna(t3)
# Convert to Boolean
df[ins_stt] = df[ins_stt].astype('boolean')
#todo Convert Object to String below

#! Select potential money cols
pattern = r'^\s*-?\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*$'
dtypes = ['string','object']
def get_money_col(df, dtypes, pattern):
    def df_select_dtypes():
        selected_df = df.select_dtypes(include=dtypes)
        return selected_df
    potential_cols = df_select_dtypes()
    def the_condition(col):
        # Lấy 500 dòng đầu, kiểm tra khớp pattern, tính trung bình
        return col.dropna().head(500).astype(str).str.match(pattern).mean() >= 0.9
    def loop_cols():
        found_cols = []
        if not potential_cols.empty:
            for each_col in potential_cols:
                if the_condition(potential_cols[each_col]):
                    # print(f"Tìm thấy cột số: {each_col}")
                    found_cols.append(each_col)
        return found_cols
    return loop_cols()
money_col = get_money_col(df, dtypes, pattern)
#! Convert money cols to numeric
def comma_to_num(df, cols_name):
    if not cols_name:
        return []
    for col in cols_name:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
    return cols_name

pd.options.display.float_format = '{:.0f}'.format
vnd = comma_to_num(df, money_col)


#! Detect dtypes and seperate specials one
all_cols = df.columns.to_list()
all_minus_special = [col for col in all_cols if col not in special_cols]

# Classifying cols by dtypes | df.select_dtypes(include='dtype') |
num_cols = df[all_minus_special].select_dtypes(include='number').columns.to_list()
num_cols = list(set(num_cols + vnd))
str_cols = df[all_minus_special].select_dtypes(include='object').columns.to_list()
datetime_cols = df[all_minus_special].select_dtypes(include='datetime').columns.to_list()
#todo Convert Object to String
df[str_cols] = df[str_cols].astype('string')

#! Fillna()
#? {**dict.fromkeys(['key'], 1)} Chỉ có thể {unpack **dict} bên trong dict || bên trong parameter của function(,**dict)
fill_rules = (
    {col: 0 for col in num_cols} | 
    {col: 'unknown' for col in str_cols} | 
    {col: pd.NaT for col in datetime_cols} | 
    {qty: 0, ins_stt: False}
    )
df_clean = df.fillna(fill_rules)


#! Dtype Optimization
types_map = {
    'bill': 'int64',
    'ean': 'string',
    qty: 'int32'}                                             #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
df_clean = df_clean.astype(types_map, errors='ignore')

# Double checking   | List + List... | Set(list) - Set(list)
accounted_cols = num_cols + str_cols + datetime_cols + special_cols
if_missing_cols = list(set(all_cols) - set(accounted_cols))

# Fail-safe/Log     | assert True, False
print(f'Money cols: {money_col}')
print(f"Số dòng thiếu Date: {df_clean[date].isna().sum()}")
print(f"Số dòng thiếu Time: {df_clean[time].isna().sum()}")
print(f"Số dòng lặp:        {df_clean.duplicated().sum()}")
print(f"Số dòng thiếu Qty:  {df_clean[qty].isna().sum()}")
print(f"Số dòng Price <= 0: {(df_clean[price]<=0).sum()}") 

assert (df_clean[qty] >= 0).all(),      "SOS: qty âm"
assert df_clean[qty].isna().sum() == 0, "SOS: Có dòng qty = None!"
assert len(if_missing_cols) == 0,        f"SOS: Check lại đi, thiếu {if_missing_cols} rồi!"

df_clean['only_time'] = df_clean['time'].dt.time
# df.dtypes
# df_clean
# df_clean['cat'].value_counts()
# df_clean.groupby('sa', observed=True)['revenue'].sum()
# df_clean.loc[df_clean['cat'] == 'unknown']
df_clean['price']
df_clean[['cat','sap_description','revenue']]
df.dtypes

Money cols: ['price', 'revenue', 'cash', 'card/zlp', 'payoo', 'banking', 'mkt', 'vnpay', 'trade-in']
Số dòng thiếu Date: 6968
Số dòng thiếu Time: 4738
Số dòng lặp:        4762
Số dòng thiếu Qty:  0
Số dòng Price <= 0: 4745


date                               datetime64[ns]
bill                               string[python]
sa                                 string[python]
ean                                       float64
cat                                string[python]
imei/sn                            string[python]
sap_article                        string[python]
sap_description                    string[python]
price                                     float64
sl                                        float64
tg                                        boolean
insfee                                    float64
revenue                                   float64
cash                                      float64
card/zlp                                  float64
payoo                                     float64
banking                                   float64
mkt                                       float64
vnpay                                     float64
trade-in                                  float64


In [ ]:
pattern = r'^\s*-?\d{1,3}(?:,\d{3})*(?:\.\d+)?\s*$'
dtypes = ['string','object']

def get_money_col(df, dtypes, pattern):
    def df_select_dtypes():
        selected_df = df.select_dtypes(include=dtypes)
        return selected_df
    potential_cols = df_select_dtypes()
    def the_condition(col):
        # Lấy 500 dòng đầu, kiểm tra khớp pattern, tính trung bình
        return col.dropna().head(500).astype(str).str.match(pattern).mean() >= 0.9
    def loop_cols():
        found_cols = []
        if not potential_cols.empty:
            for each_col in potential_cols:
                if the_condition(potential_cols[each_col]):
                    # print(f"Tìm thấy cột số: {each_col}")
                    found_cols.append(each_col)
        return found_cols
    return loop_cols()
x = get_money_col(df, dtypes, pattern)

def comma_to_num(df, cols_name):
    if not cols_name:
        return df
    for col in cols_name:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
    return df[cols_name]
print(x)
pd.options.display.float_format = '{:.0f}'.format
y = comma_to_num(df, x)



In [ ]:
import pandas as pd
import numpy as np
pd.options.display.float_format = '{:,.0f}'.format

df = pd.read_csv(r'D:\Python\LMHN_2025_DIRTY.csv')

# Standardize cols names
df.columns = [col.lower().replace(' ','_') for col in df.columns]

# Get col name from regex
def get_col(df, rule):
  # col_mask = object index[boolean] 
    col_mask = df.columns.astype(str).str.contains(rule, case=False, regex=True)
  # count_true = Count True in col_mask
    count_true = col_mask.sum()
  # Apply mask to columns_names [object]
    col = df.columns[col_mask]
     
    if   count_true == 1:     
        return col[0]
    elif count_true == 0:
        print(f'get_col 0 match for regex: {rule}')
        return None
    else:
        print(f'get_col return first_match for regex: {rule}.Found {count_true} cols: {col.to_list()} Investigate later!')
        return col[0]

# List cols types to start
drop_cols = ['ins_fee','disc_percent','disc_amount','vat']
special_cols = ['qty', 'ins_stt'] 
date = get_col(df, r'\bdate\b|\bngay\b|\bday\b')
time = get_col(df, r'\btime\b|\bgio\b|\bhour\b')
qty = get_col(df, r'\bqty\b|quantity')
price = get_col(df, r'\bprice\b')

# Drop unnecessary columns/rows. Never sort when cleaning, don't mess with labels
df = df.drop(drop_cols, axis=1, errors='ignore')


#! Need to convert date/time/string/boolean cols BEFORE Fillna()
# Date processing | pd.to_datetime(df['col_name']) |
if date:
  d1 = pd.to_datetime(df[date], format='%Y-%m-%d', errors='coerce')    # 2026-01-20
  d2 = pd.to_datetime(df[date], format='%d-%m-%Y', errors='coerce')    # 20-01-2026
  df[date] = d1.fillna(d2)
# Time processing
if time:
  t1 = pd.to_datetime(df[time], format='%I:%M%p' , errors='coerce')    # 8:30am
  t2 = pd.to_datetime(df[time], format='%H:%M:%S', errors='coerce')    # 20:30:00
  t3 = pd.to_datetime(df[time], format='%H:%M'   , errors='coerce')    # 16:20
  df[time] = t1.fillna(t2).fillna(t3)
# Convert to Boolean
df['ins_stt'] = df['ins_stt'].astype('boolean')
#todo Convert Object to String below


#! Detect dtypes and seperate specials one
all_cols = df.columns.to_list()
all_minus_special = [col for col in all_cols if col not in special_cols]

# Classifying cols by dtypes | df.select_dtypes(include='dtype') |
num_cols = df[all_minus_special].select_dtypes(include='number').columns.to_list()
str_cols = df[all_minus_special].select_dtypes(include='object').columns.to_list()
datetime_cols = df[all_minus_special].select_dtypes(include='datetime').columns.to_list()
#todo Convert Object to String
df[str_cols] = df[str_cols].astype('string')

#! Fillna()
#? {**dict.fromkeys(['key'], 1)} Chỉ có thể {unpack **dict} bên trong dict || bên trong parameter của function(,**dict)
fill_rules = (
    {col: 0 for col in num_cols} | 
    {col: 'unknown' for col in str_cols} | 
    {col: pd.NaT for col in datetime_cols} | 
    {qty: 0, 'ins_stt': False}
    )
df_clean = df.fillna(fill_rules)


#! Dtype Optimization
types_map = {
    'invoice': 'int64',
    'ean': 'string',
    qty: 'int32',
    'cat': 'category',
    'sa' : 'category'}
df_clean = df_clean.astype(types_map, errors='ignore')

# Double checking   | List + List... | Set(list) - Set(list)
accounted_cols = num_cols + str_cols + datetime_cols + special_cols
if_missing_cols = list(set(all_cols) - set(accounted_cols))

# Fail-safe/Log     | assert True, False
print(f"Số dòng thiếu Date: {df_clean[date].isna().sum()}")
print(f"Số dòng thiếu Time: {df_clean[time].isna().sum()}")
print(f"Số dòng lặp:        {df_clean.duplicated().sum()}")
print(f"Số dòng thiếu Qty:  {df_clean[qty].isna().sum()}")
print(f"Số dòng Price <= 0: {(df_clean[price]<=0).sum()}")

assert (df_clean[qty] >= 0).all(),      "SOS: qty âm"
assert df_clean[qty].isna().sum() == 0, "SOS: Có dòng qty = None!"
assert len(if_missing_cols) == 0,        f"SOS: Check lại đi, thiếu {if_missing_cols} rồi!"

df_clean['only_time'] = df_clean['time'].dt.time
# df.dtypes
# df_clean
# df_clean['cat'].value_counts()
# df_clean.groupby('sa', observed=True)['revenue'].sum()
# df_clean.loc[df_clean['cat'] == 'unknown']

In [ ]:
q = {'mickey': 0} 
q['mickey'] = 1990
q
bd = 'Col1'

if 'col' in bd.lower():
    print('Oke')
else:
    print('Noooo')
# .str.string_methods chỉ dùng đc với các object 1 chiều VD: dãy tên cột, dãy Label, 1 Series trong DF
# Lại còn phải gọi đi gọi lại .str nếu dùng các method khác nhau
# n = df.columns.astype(str).str.contains(r'\bdate\b|\bngay\b', case=False, regex=True)
# date = df.loc[:, n if df.loc[:,n].columns.nunique() == 1 else n]
#! auto_date
the_date = df.filter(regex=r'(?i)\bdate\b|\bngay\b|\bday\b')
the_time = df.loc[:,df.columns.astype(str).str.contains(r'\btime\b|\bgio\b|\bhour\b', case=False, regex=True)]

assert len(the_date.columns) <= 1,'SOS: date problem'
assert len(the_time.columns) <= 1,'SOS: time problem'

date = the_date.columns[0] if len(the_date.columns) == 1 else None
time = the_time.columns[0] if len(the_time.columns) == 1 else None

Oke


In [75]:
def get_col(df, rule):
  # col_mask = object index[boolean] 
    col_mask = df.columns.astype(str).str.contains(rule, case=False, regex=True)
  # count_true = Count True in col_mask
    count_true = col_mask.sum()
  # Apply mask to columns_names [object]
    col = df.columns[col_mask]
     
    if   count_true == 1:     
        return col[0]
    elif count_true == 0:
        print(f'get_col 0 match for regex: {rule}')
        return None
    else:
        print(f'get_col return first_match for regex: {rule}.Found {count_true} cols: {col.to_list()} Investigate later!')
        return col[0]
   

date = get_col(df,r'\btime\b|\bgio\b|\bhour\b')
if None:
    print(True)
else:
    print(False)


False
